# SSSD-ECG training — sex-skewed subset (25% female / 75% male)

Same pipeline as `Train_custum_SSSD_ECG.ipynb`, except the generator train subset is built by:

- Fixed size **N** = same as the original **10% of the PTB-XL train fold** (~1741).
- Target sex ratio **25% female / 75% male**.
- Within each sex, **MI prevalence** matches the **full train split** (`df_train_real`).
- Integer counts for the four strata (male/female × MI/no-MI), then **sampling without replacement** per stratum with **`SUBSET_SEED`**.

Change **`SUBSET_SEED`** when you want a second independent subset; row indices are saved under Drive for later classifier training.

**Run the balanced-sex notebook in a separate session** (or after training) so `ptbxl_train_data.npy` in `sssd/` is not overwritten mid-run.

In [ ]:
!pip install -q pykeops

In [ ]:
# Cell 1: basic environment + repo setup
import os
import sys
from pathlib import Path

!pip install -q wfdb resampy ishneholterlib pytorch-lightning

REPO_URL = "https://github.com/Anonymous0002396/CMED-Mini-Project.git"
PROJECT_ROOT = Path("/content/CMED-Mini-Project")
REPO_ROOT = PROJECT_ROOT / "SSSD-ECG"

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} /content/CMED-Mini-Project
else:
    %cd /content/CMED-Mini-Project
    !git pull

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "src" / "ptb_xl"))
sys.path.insert(0, str(REPO_ROOT / "src" / "sssd"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPO_ROOT exists:", REPO_ROOT.exists())
print("SSSD train.py exists:", (REPO_ROOT / "src" / "sssd" / "train.py").exists())
print("SSSD model exists:", (REPO_ROOT / "src" / "sssd" / "models" / "SSSD_ECG.py").exists())

In [ ]:
# Cell 2: mount Drive and extract raw PTB-XL
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

if not os.path.exists('/content/ptbxl.zip'):
    !cp "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.zip" /content/ptbxl.zip

if not os.path.exists('/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1'):
    !unzip -oq /content/ptbxl.zip -d /content/
    print("Extraction complete.")
else:
    print("PTB-XL already extracted, skipping extraction.")

raw_ptbxl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")
print("raw_ptbxl exists:", raw_ptbxl.exists())

In [ ]:
# Cell 3: preprocess PTB-XL at 100 Hz and load processed metadata
import numpy as np
from pathlib import Path

from clinical_ts.ecg_utils import prepare_data_ptb_xl, channel_stoi_default
from clinical_ts.timeseries_utils import reformat_as_memmap, load_dataset

target_fs = 100
data_folder_ptb_xl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")
target_folder_ptb_xl = Path(f"/content/processed_ptb_xl_fs{target_fs}")

# Force rebuild for a clean notebook run
if target_folder_ptb_xl.exists():
    !rm -rf {target_folder_ptb_xl}

print("Rebuilding processed PTB-XL at:", target_folder_ptb_xl)

df_ptb_xl, lbl_itos_ptb_xl, mean_ptb_xl, std_ptb_xl = prepare_data_ptb_xl(
    data_path=data_folder_ptb_xl,
    min_cnt=0,
    target_fs=target_fs,
    channels=12,
    channel_stoi=channel_stoi_default,
    target_folder=target_folder_ptb_xl,
    recreate_data=True,
)

reformat_as_memmap(
    df_ptb_xl,
    target_folder_ptb_xl / "memmap.npy",
    data_folder=target_folder_ptb_xl,
    delete_npys=True
)

df_mapped, lbl_itos_dict, mean, std = load_dataset(target_folder_ptb_xl)

print("Processed df shape:", df_mapped.shape)
print("Available label spaces:", list(lbl_itos_dict.keys()))
print("Processed folder:", target_folder_ptb_xl)

In [ ]:
# Cell 4: create sex + MI labels on the full processed dataframe
import numpy as np
import pandas as pd

label_key = "label_all"
label_names = np.array(lbl_itos_dict[label_key])

mi_statement_names = [
    "IMI", "ASMI", "ILMI", "AMI", "ALMI",
    "INJAS", "LMI", "INJAL", "IPLMI", "IPMI",
    "INJIN", "PMI", "INJLA", "INJIL"
]

label_name_to_idx = {str(name): i for i, name in enumerate(label_names)}
mi_label_indices = [label_name_to_idx[name] for name in mi_statement_names]

def multihot_encode(indices, num_classes):
    out = np.zeros(num_classes, dtype=np.float32)
    for i in indices:
        out[i] = 1.0
    return out

def row_multihot_to_binary_mi(row_vec, mi_indices):
    return float(row_vec[mi_indices].sum() > 0)

df_real = df_mapped.copy()

df_real["label_multihot"] = df_real[f"{label_key}_numeric"].apply(
    lambda x: multihot_encode(x, len(label_names))
)

df_real["label_mi"] = df_real["label_multihot"].apply(
    lambda x: row_multihot_to_binary_mi(x, mi_label_indices)
).astype(np.float32)

df_real["sex_binary"] = df_real["sex"].astype(np.float32)

print("Unique sex values:", sorted(df_real["sex_binary"].dropna().unique().tolist()))
print("Overall MI prevalence:", df_real["label_mi"].mean(), int(df_real["label_mi"].sum()), len(df_real))
print("\nCounts by (sex, MI):")
print(df_real.groupby(["sex_binary", "label_mi"]).size())

In [ ]:
# Cell 5: train/val/test split + N = same count as original 10% stratified draw
max_fold_id = df_real["strat_fold"].max()

df_train_real = df_real[df_real["strat_fold"] < max_fold_id - 1].copy()
df_val_real = df_real[df_real["strat_fold"] == max_fold_id - 1].copy()
df_test_real = df_real[df_real["strat_fold"] == max_fold_id].copy()

print("Full split sizes:")
print("train:", len(df_train_real))
print("val:  ", len(df_val_real))
print("test: ", len(df_test_real))

# Match original notebook: 10% of train fold → same N as stratified 10% split
TRAIN_SUBSET_FRAC = 0.10
N_TRAIN_SUBSET = max(1, int(round(TRAIN_SUBSET_FRAC * len(df_train_real))))
print("\nTarget generator/classifier train subset size N:", N_TRAIN_SUBSET, "(10% of train fold)")

In [ ]:
# Cell 6: sex-skewed subset — 25% female / 75% male, within-sex MI rate = full train
import numpy as np

# --- edit SUBSET_SEED for a second independent run ---
SUBSET_SEED = 42
FEMALE_FRACTION = 0.25

rng = np.random.default_rng(SUBSET_SEED)

male_mask = df_train_real["sex_binary"] == 0.0
female_mask = df_train_real["sex_binary"] == 1.0
n_male_pool = int(male_mask.sum())
n_female_pool = int(female_mask.sum())

p_mi_male = float(df_train_real.loc[male_mask, "label_mi"].mean())
p_mi_female = float(df_train_real.loc[female_mask, "label_mi"].mean())

print("Full train fold: P(MI | male)  =", p_mi_male)
print("Full train fold: P(MI | female)=", p_mi_female)

n_female = int(round(N_TRAIN_SUBSET * FEMALE_FRACTION))
n_male = N_TRAIN_SUBSET - n_female
print("\nTarget counts: n_male =", n_male, ", n_female =", n_female, ", sum =", n_male + n_female)


def _mi_counts_for_sex(n_sex, p_mi):
    """Return (n_no_mi, n_mi) for one sex, summing to n_sex."""
    n_mi = int(round(n_sex * p_mi))
    n_mi = max(0, min(n_sex, n_mi))
    n_no = n_sex - n_mi
    return n_no, n_mi


m0, m1 = _mi_counts_for_sex(n_male, p_mi_male)
f0, f1 = _mi_counts_for_sex(n_female, p_mi_female)

targets = {
    (0.0, 0.0): m0,
    (0.0, 1.0): m1,
    (1.0, 0.0): f0,
    (1.0, 1.0): f1,
}
print("\nTarget strata (sex, MI) -> count:", targets)
print("Sum:", sum(targets.values()), "(expect", N_TRAIN_SUBSET, ")")

pools = {}
for sex in (0.0, 1.0):
    for mi in (0.0, 1.0):
        mask = (df_train_real["sex_binary"] == sex) & (df_train_real["label_mi"] == mi)
        pools[(sex, mi)] = df_train_real.index[mask].to_numpy()

print("\nPool sizes (sex, MI):", {k: len(v) for k, v in pools.items()})

picked_idx = []
for key, k_need in targets.items():
    pool = pools[key]
    if k_need > len(pool):
        raise RuntimeError(
            f"Not enough samples for stratum {key}: need {k_need}, pool {len(pool)}"
        )
    if k_need > 0:
        take = rng.choice(pool, size=k_need, replace=False)
        picked_idx.append(take)

train_subset_idx = np.sort(np.concatenate(picked_idx))
df_train_subset = df_train_real.loc[train_subset_idx].copy().sort_index()

print("\nActual subset size:", len(df_train_subset))
print("Female fraction:", (df_train_subset["sex_binary"] == 1.0).mean())
print("MI | male (subset):", df_train_subset[df_train_subset["sex_binary"] == 0.0]["label_mi"].mean())
print("MI | female (subset):", df_train_subset[df_train_subset["sex_binary"] == 1.0]["label_mi"].mean())
print("\nCounts by (sex, MI):")
print(df_train_subset.groupby(["sex_binary", "label_mi"]).size())

In [ ]:
# Cell 7: align memmap dataframe and attach sex/MI conditioning
import pickle
import numpy as np

with open(target_folder_ptb_xl / "df_memmap.pkl", "rb") as f:
    df_memmap_meta = pickle.load(f)

df_memmap_meta = df_memmap_meta.copy()

df_memmap_meta["sex_binary"] = df_real["sex_binary"].values.astype(np.float32)
df_memmap_meta["label_mi"] = df_real["label_mi"].values.astype(np.float32)
df_memmap_meta["strat_fold"] = df_real["strat_fold"].values

cond_matrix = np.stack(
    [
        df_memmap_meta["sex_binary"].to_numpy(dtype=np.float32),
        df_memmap_meta["label_mi"].to_numpy(dtype=np.float32),
    ],
    axis=1,
).astype(np.float32)

df_memmap_meta["cond_sex_mi"] = list(cond_matrix)

print("df_memmap_meta shape:", df_memmap_meta.shape)
print("Counts by (sex, MI):")
print(df_memmap_meta.groupby(["sex_binary", "label_mi"]).size())

In [ ]:
# Cell 8: memmap-backed split dataframes (generator train = same rows as subset)
max_fold_id = df_memmap_meta["strat_fold"].max()

df_val_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id - 1].copy()
df_test_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id].copy()

df_train_real_mem_subset = df_memmap_meta.loc[df_train_subset.index].copy()

print("Memmap split sizes:")
print("train subset:", len(df_train_real_mem_subset))
print("val:         ", len(df_val_real_mem))
print("test:        ", len(df_test_real_mem))
print("\nSubset counts by (sex, MI):")
print(df_train_real_mem_subset.groupby(["sex_binary", "label_mi"]).size())

In [ ]:
# Cell 9: memmap-backed datasets with 2-d condition vectors
from clinical_ts.timeseries_utils import TimeseriesDatasetCrops, ToTensor

input_size = 1000
tfms_ptb_xl = ToTensor()

train_real_ds_subset = TimeseriesDatasetCrops(
    df_train_real_mem_subset,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl="cond_sex_mi",
    memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

val_real_ds_full12 = TimeseriesDatasetCrops(
    df_val_real_mem,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl="cond_sex_mi",
    memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

test_real_ds_full12 = TimeseriesDatasetCrops(
    df_test_real_mem,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl="cond_sex_mi",
    memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

sample_x, sample_cond = train_real_ds_subset[0]
print("sample ECG shape:", sample_x.shape)
print("sample cond:", sample_cond)
print("train len:", len(train_real_ds_subset))
print("val len:", len(val_real_ds_full12))
print("test len:", len(test_real_ds_full12))

In [ ]:
# Cell 10: materialize train subset into NumPy arrays
train_data_list = []
train_cond_list = []

for i in range(len(train_real_ds_subset)):
    x, cond = train_real_ds_subset[i]
    train_data_list.append(x.numpy().astype(np.float32))
    train_cond_list.append(np.asarray(cond, dtype=np.float32))

train_data = np.stack(train_data_list).astype(np.float32)
train_cond = np.stack(train_cond_list).astype(np.float32)

print("train_data shape:", train_data.shape, train_data.dtype)
print("train_cond shape:", train_cond.shape, train_cond.dtype)

In [ ]:
# Cell 11: save arrays for train.py + persist row indices (classifier / reproducibility)
from pathlib import Path

sssd_workdir = REPO_ROOT / "src" / "sssd"
sssd_workdir.mkdir(parents=True, exist_ok=True)

# train.py hardcodes these names — always write here before training
np.save(sssd_workdir / "ptbxl_train_data.npy", train_data)
np.save(sssd_workdir / "ptbxl_train_labels.npy", train_cond)

row_indices = df_train_subset.index.to_numpy()
np.save(sssd_workdir / "generator_train_row_indices.npy", row_indices)

print("Saved:", sssd_workdir / "ptbxl_train_data.npy")
print("Saved:", sssd_workdir / "ptbxl_train_labels.npy")
print("Saved row indices:", sssd_workdir / "generator_train_row_indices.npy", "shape", row_indices.shape)

# Per-arm backup copies (optional archive)
# np.save(sssd_workdir / "ptbxl_train_data_skewed_25f_75m.npy", train_data)
# np.save(sssd_workdir / "ptbxl_train_labels_skewed_25f_75m.npy", train_cond)

DRIVE_SUBSET_DIR = Path("/content/drive/MyDrive/sssd_sex_mi_skewed_25f_75m_subsets")
DRIVE_SUBSET_DIR.mkdir(parents=True, exist_ok=True)
np.save(DRIVE_SUBSET_DIR / f"generator_train_row_indices_seed{SUBSET_SEED}.npy", row_indices)
meta = {
    "subset_seed": int(SUBSET_SEED),
    "female_fraction": FEMALE_FRACTION,
    "N": int(len(row_indices)),
    "p_mi_male_full_train": p_mi_male,
    "p_mi_female_full_train": p_mi_female,
}
import json
with open(DRIVE_SUBSET_DIR / f"subset_meta_seed{SUBSET_SEED}.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Drive index + meta:", DRIVE_SUBSET_DIR)




In [ ]:
np.save(DRIVE_SUBSET_DIR / f"ptbxl_train_data_seed{SUBSET_SEED}.npy", train_data)
np.save(DRIVE_SUBSET_DIR / f"ptbxl_train_labels_seed{SUBSET_SEED}.npy", train_cond)

In [ ]:
# Cell 12: JSON config (separate checkpoint dir for this arm)
import json
from pathlib import Path

sssd_workdir = REPO_ROOT / "src" / "sssd"
config_dir = sssd_workdir / "config"
config_dir.mkdir(parents=True, exist_ok=True)

cfg_path = config_dir / "config_SSSD_ECG_sex_mi_skewed_25f_75m.json"
drive_output_dir = "/content/drive/MyDrive/sssd_sex_mi_skewed_25f_75m"

sex_mi_cfg = {
    "diffusion_config": {
        "T": 200,
        "beta_0": 0.0001,
        "beta_T": 0.02
    },
    "wavenet_config": {
        "in_channels": 8,
        "out_channels": 8,
        "num_res_layers": 36,
        "res_channels": 256,
        "skip_channels": 256,
        "diffusion_step_embed_dim_in": 128,
        "diffusion_step_embed_dim_mid": 512,
        "diffusion_step_embed_dim_out": 512,
        "s4_lmax": 1000,
        "s4_d_state": 64,
        "s4_dropout": 0.0,
        "s4_bidirectional": 1,
        "s4_layernorm": 1,
        "label_embed_dim": 128,
        "label_embed_classes": 2
    },
    "train_config": {
        "output_directory": drive_output_dir,
        "ckpt_iter": "max",
        "iters_per_ckpt": 2000,
        "iters_per_logging": 200,
        "n_iters": 15000,
        "learning_rate": 2e-4,
        "batch_size": 8
    },
    "trainset_config": {
        "segment_length": 1000,
        "sampling_rate": 100,
        "finetune_dataset": "ptbxl_sex_mi_skewed_25f_75m"
    },
    "gen_config": {
        "output_directory": drive_output_dir,
        "ckpt_path": drive_output_dir
    }
}

with open(cfg_path, "w") as f:
    json.dump(sex_mi_cfg, f, indent=2)

print("Wrote:", cfg_path)
print(json.dumps(sex_mi_cfg, indent=2))

Patch `train.py` so `DataLoader` uses `batch_size` from the config (same as original notebook).

In [ ]:
from pathlib import Path

train_py = REPO_ROOT / "src" / "sssd" / "train.py"
lines = train_py.read_text().splitlines()
for i, line in enumerate(lines):
    if "trainloader" in line and "DataLoader" in line:
        print(f"LINE {i}: {repr(line)}")

In [ ]:
train_py = REPO_ROOT / "src" / "sssd" / "train.py"
text = train_py.read_text()

if "batch_size=batch_size" in text:
    print("Batch size already patched.")
else:
    if "batch_size=6" not in text:
        raise RuntimeError("Could not find 'batch_size=6' anywhere in train.py")
    text = text.replace("batch_size=6", "batch_size=batch_size")
    train_py.write_text(text)
    print("Patched batch size in:", train_py)

### Train model

Uncomment the next cell to launch training (runs from `sssd/` so `ptbxl_train_data.npy` resolves).

In [ ]:
from pathlib import Path

ckpt_dir = Path("/content/drive/MyDrive/sssd_sex_mi_skewed_25f_75m/ch256_T200_betaT0.02")
print("exists:", ckpt_dir.exists())
if ckpt_dir.exists():
    print(sorted([p.name for p in ckpt_dir.glob("*.pkl")])[-10:])

In [ ]:
import pykeops
print(pykeops.__version__)

In [ ]:
# Train sex+MI-conditioned SSSD on the skewed subset
%cd /content/CMED-Mini-Project/SSSD-ECG/src/sssd
!python train.py -c config/config_SSSD_ECG_sex_mi_skewed_25f_75m.json